In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Geometry-V4 G0 detached subprocess handoff
The current notebook kernel never imports project packages. It installs the detached checkout, then starts one fresh Python subprocess for the frozen G0 runner. Drive is a create-only result sink.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib, json, subprocess, sys

REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
BRANCH = 'Geometry-V4'
SOURCE_EXACT = 'a4f8d5c1492c7e7fc14207e394a0f4714756c178'
REPO = Path('/content/cegwm-geometry-v4-g0-g1-source')
DRIVE_RUNS = Path('/content/drive/MyDrive/CEG-WM/Geometry-V4-G0-G1/runs')

if REPO.exists():
    raise FileExistsError('fresh Colab runtime required; source checkout already exists')
subprocess.run(['git', 'clone', '--single-branch', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
def git(*args):
    return subprocess.run(['git', *args], cwd=REPO, check=True, capture_output=True, text=True).stdout.strip()

CLONED_BRANCH = git('branch', '--show-current')
subprocess.run(['git', 'checkout', '--detach', SOURCE_EXACT], cwd=REPO, check=True)
assert git('rev-parse', 'HEAD') == SOURCE_EXACT
assert git('branch', '--show-current') == ''
assert git('status', '--porcelain') == ''
CONFIG_SHA256 = hashlib.sha256((REPO/'configs/geometry_v4/geometry_v4_g0_g1_v1.json').read_bytes()).hexdigest()
RUN_UTC = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_ROOT = DRIVE_RUNS / f'{SOURCE_EXACT}-{RUN_UTC}'
DRIVE_RUNS.mkdir(parents=True, exist_ok=True)
if RUN_ROOT.exists():
    raise FileExistsError('create-only run directory already exists')
print({'branch': CLONED_BRANCH, 'source_exact': SOURCE_EXACT, 'detached': True, 'clean': True, 'config_sha256': CONFIG_SHA256, 'drive_run_root': str(RUN_ROOT)})


In [ ]:
import os, torch
from google.colab import userdata
assert torch.cuda.is_available(), 'GPU runtime is required; no G0 record may be created on CPU'
gpu = {'name': torch.cuda.get_device_name(0), 'vram_bytes': torch.cuda.get_device_properties(0).total_memory}
print({'cuda_available': True, 'gpu': gpu})
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(REPO)], check=True)
root_key = userdata.get('CEG_WM_ROOT_KEY')
hf_token = userdata.get('HF_TOKEN')
assert all(isinstance(value, str) and value.strip() for value in (root_key, hf_token))
secret_markers = ('TOKEN', 'KEY', 'SECRET', 'PASSWORD', 'CREDENTIAL')
runner_env = {name: value for name, value in os.environ.items() if not any(marker in name.upper() for marker in secret_markers)}
runner_env['CEG_WM_ROOT_KEY'] = root_key
runner_env['HF_TOKEN'] = hf_token
root_key = ''
hf_token = ''
command = [sys.executable, '-m', 'experiments.geometry_v4_generative_engine', '--stage', 'G0', '--repo-root', str(REPO), '--artifact-root', str(RUN_ROOT), '--expected-exact', SOURCE_EXACT]
try:
    completed = subprocess.run(command, cwd=REPO, env=runner_env, text=True, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=False)
finally:
    runner_env.pop('CEG_WM_ROOT_KEY', None)
    runner_env.pop('HF_TOKEN', None)
    runner_env = None
summary = completed.stdout[-8192:]
if completed.returncode != 0:
    raise RuntimeError('Geometry-V4 G0 subprocess stopped: ' + summary)
print(summary)
# No retry, resume, fallback, or G1 invocation in this notebook.
